# Multi-Horizon Prediction Experiment

The XGBoost notebook showed that next-day prediction is a near-random-walk problem where adding complexity only worsens results. This experiment tests whether **longer horizons** and **directional classification** perform better:

1. **Multi-horizon targets** -- 5, 10, 20, 30-day ahead returns and direction
2. **Dimensionality reduction** -- address curse of dimensionality (71 features vs ~207 samples) via variance-based filtering, correlation pruning, and feature selection
3. **Regression** -- XGBoost regressor for N-day return prediction
4. **Classification** -- XGBoost classifier for up/down direction
5. **GridSearchCV** with TimeSeriesSplit for each horizon
6. **Walk-forward validation** -- same expanding window framework

**Input:** `features_price.csv`, `features_nlp.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             accuracy_score, f1_score, classification_report)
from sklearn.feature_selection import mutual_info_regression
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = '..'
RANDOM_STATE = 42
TRAIN_PCT = 0.7
STEP_SIZE = 5
HORIZONS = [5, 10, 20, 30]

np.random.seed(RANDOM_STATE)
print(f'Horizons: {HORIZONS} days')
print(f'Train split: {TRAIN_PCT}')

## 1. Load and Merge Features

In [ ]:
price_features = pd.read_csv(f'{DATA_DIR}/features_price.csv', parse_dates=['date'])
nlp_features = pd.read_csv(f'{DATA_DIR}/features_nlp.csv', parse_dates=['date'])

df = price_features.merge(nlp_features, on=['date', 'ticker'], how='left')
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

nlp_cols = [c for c in nlp_features.columns if c not in ['date', 'ticker']]
for col in nlp_cols:
    if col == 'has_news':
        df[col] = df[col].fillna(0).astype(int)
    else:
        df[col] = df[col].fillna(0.0)

tickers = sorted(df['ticker'].unique())

META_COLS = ['date', 'ticker']
TARGET_COLS = ['target_next_close', 'target_next_return', 'target_direction']
OHLCV_COLS = ['open', 'high', 'low', 'close', 'volume']
EXCLUDE_COLS = META_COLS + TARGET_COLS + ['is_outlier']

price_feature_cols = [c for c in price_features.columns if c not in EXCLUDE_COLS + OHLCV_COLS]
nlp_feature_cols = list(nlp_cols)
all_feature_cols = price_feature_cols + nlp_feature_cols

print(f'Merged shape: {df.shape}')
print(f'Total features before reduction: {len(all_feature_cols)}')
print(f'Tickers: {tickers}')
print(f'Samples per ticker: ~{len(df) // len(tickers)}')

## 2. Create Multi-Horizon Targets

For each horizon N, create:
- **Return**: `(close[t+N] - close[t]) / close[t]` -- percentage return over N days
- **Direction**: 1 if return > 0, else 0 -- binary up/down classification

In [ ]:
for h in HORIZONS:
    df[f'target_return_{h}d'] = df.groupby('ticker')['close'].transform(
        lambda x: x.shift(-h) / x - 1
    )
    df[f'target_dir_{h}d'] = (df[f'target_return_{h}d'] > 0).astype(int)

print('Multi-horizon targets created:')
for h in HORIZONS:
    valid = df[f'target_return_{h}d'].notna().sum()
    up_pct = df[f'target_dir_{h}d'].mean() * 100
    print(f'  {h:>2}d: {valid} valid samples, '
          f'mean return = {df[f"target_return_{h}d"].mean()*100:.2f}%, '
          f'up ratio = {up_pct:.1f}%')

## 3. Dimensionality Reduction

With ~207 samples per ticker and 71 features, we face the **curse of dimensionality**. Strategy:

1. **Remove near-zero variance** features
2. **Remove highly correlated** feature pairs (keep the one with higher target correlation)
3. Report the reduced feature set size

In [ ]:
sample_df = df.dropna(subset=[f'target_return_{HORIZONS[0]}d']).copy()

# Step 1: Remove near-zero variance features
variances = sample_df[all_feature_cols].var()
low_var_threshold = 1e-6
low_var_cols = variances[variances < low_var_threshold].index.tolist()
print(f'Step 1: {len(low_var_cols)} near-zero variance features removed: {low_var_cols}')

candidate_cols = [c for c in all_feature_cols if c not in low_var_cols]

# Step 2: Remove highly correlated pairs (|r| > 0.90)
corr_matrix = sample_df[candidate_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr_pairs = []
for col in upper_tri.columns:
    correlated = upper_tri.index[upper_tri[col] > 0.90].tolist()
    for c2 in correlated:
        high_corr_pairs.append((col, c2, corr_matrix.loc[col, c2]))

print(f'\nStep 2: Found {len(high_corr_pairs)} highly correlated pairs (|r| > 0.90):')
for f1, f2, r in sorted(high_corr_pairs, key=lambda x: -x[2])[:15]:
    print(f'  {f1:<30} <-> {f2:<30} r={r:.3f}')

# For each correlated pair, drop the one with lower mean target correlation
target_col_ref = f'target_return_{HORIZONS[0]}d'
target_corrs = sample_df[candidate_cols].corrwith(sample_df[target_col_ref]).abs()

cols_to_drop = set()
for f1, f2, r in high_corr_pairs:
    if f1 in cols_to_drop or f2 in cols_to_drop:
        continue
    if target_corrs.get(f1, 0) >= target_corrs.get(f2, 0):
        cols_to_drop.add(f2)
    else:
        cols_to_drop.add(f1)

print(f'\n  Dropping {len(cols_to_drop)} redundant features: {sorted(cols_to_drop)}')

reduced_feature_cols = [c for c in candidate_cols if c not in cols_to_drop]

print(f'\nFeatures: {len(all_feature_cols)} -> {len(reduced_feature_cols)} '
      f'({len(all_feature_cols) - len(reduced_feature_cols)} removed)')
print(f'Samples per ticker: ~{len(sample_df) // len(tickers)}')
print(f'Feature-to-sample ratio: {len(reduced_feature_cols) / (len(sample_df) // len(tickers)):.2f}')

## 4. Walk-Forward Framework

In [ ]:
def walk_forward_regression(ticker_df, feature_cols, target_col,
                            model_fn, train_pct=TRAIN_PCT, step_size=STEP_SIZE,
                            verbose=True, ticker_name=''):
    """Walk-forward for regression. Returns DataFrame with date, actual, predicted."""
    sub = ticker_df.dropna(subset=[target_col]).copy().reset_index(drop=True)
    n = len(sub)
    initial_train = int(n * train_pct)
    total_steps = (n - initial_train + step_size - 1) // step_size

    X = sub[feature_cols].values
    y = sub[target_col].values
    dates = sub['date'].values

    results = []
    train_end = initial_train
    step_num = 0

    while train_end < n:
        step_num += 1
        predict_end = min(train_end + step_size, n)

        model = model_fn()
        model.fit(X[:train_end], y[:train_end])
        preds = model.predict(X[train_end:predict_end])

        for d, actual, pred in zip(dates[train_end:predict_end],
                                    y[train_end:predict_end], preds):
            results.append({'date': d, 'actual': actual, 'predicted': pred})

        if verbose and (step_num % 4 == 0 or step_num == 1 or predict_end >= n):
            print(f'  {ticker_name} step {step_num}/{total_steps} train={train_end}')

        train_end = predict_end

    return pd.DataFrame(results)


def walk_forward_classification(ticker_df, feature_cols, target_col,
                                model_fn, train_pct=TRAIN_PCT, step_size=STEP_SIZE,
                                verbose=True, ticker_name=''):
    """Walk-forward for classification. Returns DataFrame with date, actual, predicted, proba."""
    sub = ticker_df.dropna(subset=[target_col]).copy().reset_index(drop=True)
    n = len(sub)
    initial_train = int(n * train_pct)
    total_steps = (n - initial_train + step_size - 1) // step_size

    X = sub[feature_cols].values
    y = sub[target_col].values
    dates = sub['date'].values

    results = []
    train_end = initial_train
    step_num = 0

    while train_end < n:
        step_num += 1
        predict_end = min(train_end + step_size, n)

        model = model_fn()
        model.fit(X[:train_end], y[:train_end])
        preds = model.predict(X[train_end:predict_end])
        probas = model.predict_proba(X[train_end:predict_end])[:, 1]

        for d, actual, pred, prob in zip(dates[train_end:predict_end],
                                          y[train_end:predict_end], preds, probas):
            results.append({'date': d, 'actual': int(actual), 'predicted': int(pred), 'proba': prob})

        if verbose and (step_num % 4 == 0 or step_num == 1 or predict_end >= n):
            print(f'  {ticker_name} step {step_num}/{total_steps} train={train_end}')

        train_end = predict_end

    return pd.DataFrame(results)


def regression_metrics(results_df):
    actual = results_df['actual'].values
    predicted = results_df['predicted'].values
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    actual_dir = actual > 0
    pred_dir = predicted > 0
    dir_acc = np.mean(actual_dir == pred_dir) * 100
    return {'RMSE': rmse, 'MAE': mae, 'Dir_Acc': dir_acc}


def classification_metrics(results_df):
    actual = results_df['actual'].values
    predicted = results_df['predicted'].values
    acc = accuracy_score(actual, predicted) * 100
    f1 = f1_score(actual, predicted, zero_division=0) * 100
    return {'Accuracy': acc, 'F1': f1}


print('Walk-forward framework ready (regression + classification).')

## 5. GridSearchCV -- Hyperparameter Optimization Per Horizon

Run GridSearch separately for regression and classification, using pooled training data across tickers.

In [ ]:
param_grid = {
    'max_depth': [3, 4, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 300],
    'subsample': [0.7, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'min_child_weight': [3, 5],
}

n_combos = 1
for v in param_grid.values():
    n_combos *= len(v)

tscv = TimeSeriesSplit(n_splits=3)

best_params_reg = {}
best_params_clf = {}

for h in HORIZONS:
    print(f'\n{"=" * 60}')
    print(f'  GridSearch for {h}-day horizon ({n_combos} combos x 3 folds)')
    print(f'{"=" * 60}')

    grid_data = []
    for ticker in tickers:
        sub = df[df['ticker'] == ticker].dropna(subset=[f'target_return_{h}d']).copy()
        sub = sub.reset_index(drop=True)
        train_size = int(len(sub) * TRAIN_PCT)
        grid_data.append(sub.iloc[:train_size])

    grid_df = pd.concat(grid_data, ignore_index=True)
    X_grid = grid_df[reduced_feature_cols].values
    y_grid_reg = grid_df[f'target_return_{h}d'].values
    y_grid_clf = grid_df[f'target_dir_{h}d'].values

    print(f'  Training data: {X_grid.shape}')

    # --- Regression ---
    print(f'\n  Regression GridSearch...')
    gs_reg = GridSearchCV(
        xgb.XGBRegressor(reg_alpha=0.1, reg_lambda=1.0,
                         random_state=RANDOM_STATE, verbosity=0),
        param_grid, cv=tscv, scoring='neg_mean_squared_error',
        n_jobs=-1, verbose=2, refit=True
    )
    gs_reg.fit(X_grid, y_grid_reg)
    best_params_reg[h] = gs_reg.best_params_
    print(f'  Best regression RMSE: {np.sqrt(-gs_reg.best_score_):.6f}')
    print(f'  Best params: {gs_reg.best_params_}')

    # --- Classification ---
    print(f'\n  Classification GridSearch...')
    gs_clf = GridSearchCV(
        xgb.XGBClassifier(reg_alpha=0.1, reg_lambda=1.0,
                          random_state=RANDOM_STATE, verbosity=0,
                          eval_metric='logloss'),
        param_grid, cv=tscv, scoring='accuracy',
        n_jobs=-1, verbose=2, refit=True
    )
    gs_clf.fit(X_grid, y_grid_clf)
    best_params_clf[h] = gs_clf.best_params_
    print(f'  Best classification accuracy: {gs_clf.best_score_*100:.1f}%')
    print(f'  Best params: {gs_clf.best_params_}')

print('\n' + '=' * 60)
print('GridSearch completed for all horizons.')

## 6. Walk-Forward Evaluation -- All Horizons

For each horizon, run walk-forward with tuned hyperparameters for both regression and classification.

In [ ]:
all_reg_results = {}
all_clf_results = {}

for h in HORIZONS:
    print(f'\n{"=" * 60}')
    print(f'  Walk-Forward: {h}-day horizon')
    print(f'{"=" * 60}')

    reg_params = best_params_reg[h]
    clf_params = best_params_clf[h]

    def make_reg_model():
        return xgb.XGBRegressor(
            **reg_params, reg_alpha=0.1, reg_lambda=1.0,
            random_state=RANDOM_STATE, verbosity=0
        )

    def make_clf_model():
        return xgb.XGBClassifier(
            **clf_params, reg_alpha=0.1, reg_lambda=1.0,
            random_state=RANDOM_STATE, verbosity=0,
            eval_metric='logloss'
        )

    h_reg = {}
    h_clf = {}

    for ticker in tickers:
        sub = df[df['ticker'] == ticker].copy().reset_index(drop=True)

        # Regression
        print(f'\n  [{ticker}] Regression {h}d...')
        reg_df = walk_forward_regression(
            sub, reduced_feature_cols, f'target_return_{h}d',
            make_reg_model, ticker_name=ticker
        )
        h_reg[ticker] = {
            'predictions': reg_df,
            'metrics': regression_metrics(reg_df)
        }

        # Classification
        print(f'  [{ticker}] Classification {h}d...')
        clf_df = walk_forward_classification(
            sub, reduced_feature_cols, f'target_dir_{h}d',
            make_clf_model, ticker_name=ticker
        )
        h_clf[ticker] = {
            'predictions': clf_df,
            'metrics': classification_metrics(clf_df)
        }

        m_reg = h_reg[ticker]['metrics']
        m_clf = h_clf[ticker]['metrics']
        print(f'    Reg: RMSE={m_reg["RMSE"]:.4f} MAE={m_reg["MAE"]:.4f} Dir={m_reg["Dir_Acc"]:.1f}%')
        print(f'    Clf: Acc={m_clf["Accuracy"]:.1f}% F1={m_clf["F1"]:.1f}%')

    all_reg_results[h] = h_reg
    all_clf_results[h] = h_clf

print('\n' + '=' * 60)
print('Walk-forward completed for all horizons.')

## 7. Results -- Regression (Return Prediction)

In [ ]:
reg_rows = []
for h in HORIZONS:
    for ticker in tickers:
        m = all_reg_results[h][ticker]['metrics']
        reg_rows.append({'Horizon': f'{h}d', 'Ticker': ticker, **m})

reg_df_all = pd.DataFrame(reg_rows)

print('=== Regression: Mean Across Tickers ===')
reg_summary = reg_df_all.groupby('Horizon')[['RMSE', 'MAE', 'Dir_Acc']].mean().round(4)
reg_summary = reg_summary.reindex([f'{h}d' for h in HORIZONS])
display(reg_summary)

print('\n=== Regression: Per-Ticker Detail ===')
print(f'{"Horizon":<10} {"Ticker":<8} {"RMSE":>10} {"MAE":>10} {"Dir_Acc%":>10}')
print('-' * 50)
for h in HORIZONS:
    for ticker in tickers:
        m = all_reg_results[h][ticker]['metrics']
        print(f'{h:>2}d        {ticker:<8} {m["RMSE"]:>10.4f} {m["MAE"]:>10.4f} {m["Dir_Acc"]:>10.1f}')

## 8. Results -- Classification (Direction Prediction)

In [ ]:
clf_rows = []
for h in HORIZONS:
    for ticker in tickers:
        m = all_clf_results[h][ticker]['metrics']
        clf_rows.append({'Horizon': f'{h}d', 'Ticker': ticker, **m})

# Add naive baseline (always predict majority class)
for h in HORIZONS:
    for ticker in tickers:
        clf_df = all_clf_results[h][ticker]['predictions']
        majority = clf_df['actual'].mode()[0]
        naive_acc = (clf_df['actual'] == majority).mean() * 100
        clf_rows.append({'Horizon': f'{h}d', 'Ticker': ticker,
                         'Accuracy': naive_acc, 'F1': 0.0, 'Model': 'Naive'})

# Tag XGBoost rows
for row in clf_rows:
    if 'Model' not in row:
        row['Model'] = 'XGBoost'

clf_df_all = pd.DataFrame(clf_rows)

print('=== Classification: Mean Across Tickers ===')
clf_summary = clf_df_all.groupby(['Horizon', 'Model'])[['Accuracy', 'F1']].mean().round(1)
display(clf_summary)

print('\n=== Classification: Per-Ticker Detail (XGBoost) ===')
print(f'{"Horizon":<10} {"Ticker":<8} {"Accuracy%":>10} {"F1%":>10}')
print('-' * 40)
for h in HORIZONS:
    for ticker in tickers:
        m = all_clf_results[h][ticker]['metrics']
        print(f'{h:>2}d        {ticker:<8} {m["Accuracy"]:>10.1f} {m["F1"]:>10.1f}')

## 9. Visualizations

In [ ]:
# --- Regression: Dir accuracy by horizon ---
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

xgb_reg = reg_df_all.copy()
for ax, metric, title in zip(axes, ['RMSE', 'MAE', 'Dir_Acc'],
                              ['RMSE (lower is better)', 'MAE (lower is better)',
                               'Directional Accuracy % (higher is better)']):
    pivot = xgb_reg.pivot(index='Ticker', columns='Horizon', values=metric)
    pivot = pivot[[f'{h}d' for h in HORIZONS]]
    pivot.plot(kind='bar', ax=ax, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Ticker')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=0)
    ax.legend(fontsize=8)

plt.suptitle('Regression: Return Prediction Across Horizons', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Classification: Accuracy by horizon (XGBoost vs Naive) ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

xgb_clf = clf_df_all[clf_df_all['Model'] == 'XGBoost']
naive_clf = clf_df_all[clf_df_all['Model'] == 'Naive']

# Mean accuracy per horizon
xgb_mean = xgb_clf.groupby('Horizon')['Accuracy'].mean().reindex([f'{h}d' for h in HORIZONS])
naive_mean = naive_clf.groupby('Horizon')['Accuracy'].mean().reindex([f'{h}d' for h in HORIZONS])

x_pos = np.arange(len(HORIZONS))
width = 0.35
axes[0].bar(x_pos - width/2, naive_mean.values, width, label='Naive (majority)', color='gray', alpha=0.7)
axes[0].bar(x_pos + width/2, xgb_mean.values, width, label='XGBoost', color='steelblue', alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels([f'{h}d' for h in HORIZONS])
axes[0].set_xlabel('Horizon')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Classification Accuracy: XGBoost vs Naive', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].axhline(50, color='red', linestyle='--', alpha=0.4, label='Random')

# Per-ticker accuracy for best horizon
best_h = HORIZONS[xgb_mean.values.argmax()]
best_xgb = xgb_clf[xgb_clf['Horizon'] == f'{best_h}d']
best_naive = naive_clf[naive_clf['Horizon'] == f'{best_h}d']

x_pos2 = np.arange(len(tickers))
axes[1].bar(x_pos2 - width/2, best_naive['Accuracy'].values, width,
            label='Naive', color='gray', alpha=0.7)
axes[1].bar(x_pos2 + width/2, best_xgb['Accuracy'].values, width,
            label='XGBoost', color='steelblue', alpha=0.85)
axes[1].set_xticks(x_pos2)
axes[1].set_xticklabels(tickers)
axes[1].set_xlabel('Ticker')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title(f'Best Horizon ({best_h}d) -- Per-Ticker Breakdown', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].axhline(50, color='red', linestyle='--', alpha=0.4)

plt.suptitle('Classification: Direction Prediction', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Horizon comparison: mean metrics trend ---
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

reg_means = reg_df_all.groupby('Horizon')[['RMSE', 'MAE', 'Dir_Acc']].mean()
reg_means = reg_means.reindex([f'{h}d' for h in HORIZONS])

clf_xgb_means = xgb_clf.groupby('Horizon')[['Accuracy', 'F1']].mean()
clf_xgb_means = clf_xgb_means.reindex([f'{h}d' for h in HORIZONS])

axes[0].plot(range(len(HORIZONS)), reg_means['Dir_Acc'].values, 'bo-', linewidth=2, label='Regression Dir_Acc')
axes[0].plot(range(len(HORIZONS)), clf_xgb_means['Accuracy'].values, 'rs-', linewidth=2, label='Classification Acc')
axes[0].axhline(50, color='gray', linestyle='--', alpha=0.5, label='Random')
axes[0].set_xticks(range(len(HORIZONS)))
axes[0].set_xticklabels([f'{h}d' for h in HORIZONS])
axes[0].set_xlabel('Horizon')
axes[0].set_ylabel('Accuracy / Dir_Acc (%)')
axes[0].set_title('Directional Performance vs Horizon', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

axes[1].plot(range(len(HORIZONS)), reg_means['RMSE'].values, 'go-', linewidth=2)
axes[1].set_xticks(range(len(HORIZONS)))
axes[1].set_xticklabels([f'{h}d' for h in HORIZONS])
axes[1].set_xlabel('Horizon')
axes[1].set_ylabel('RMSE')
axes[1].set_title('Return Prediction RMSE vs Horizon', fontsize=12, fontweight='bold')

axes[2].plot(range(len(HORIZONS)), clf_xgb_means['F1'].values, 'mo-', linewidth=2)
axes[2].set_xticks(range(len(HORIZONS)))
axes[2].set_xticklabels([f'{h}d' for h in HORIZONS])
axes[2].set_xlabel('Horizon')
axes[2].set_ylabel('F1 Score (%)')
axes[2].set_title('Classification F1 vs Horizon', fontsize=12, fontweight='bold')

plt.suptitle('Performance Trends Across Horizons', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Best Hyperparameters Summary

In [ ]:
print('=== Best Hyperparameters Per Horizon ===')
print(f'\n{"Horizon":<10} {"Task":<15} {"max_depth":>10} {"lr":>8} {"n_est":>8} '
      f'{"subsample":>10} {"colsample":>10} {"min_cw":>8}')
print('-' * 82)

for h in HORIZONS:
    for task, params in [('Regression', best_params_reg[h]), ('Classification', best_params_clf[h])]:
        print(f'{h:>2}d        {task:<15} {params["max_depth"]:>10} '
              f'{params["learning_rate"]:>8} {params["n_estimators"]:>8} '
              f'{params["subsample"]:>10} {params["colsample_bytree"]:>10} '
              f'{params["min_child_weight"]:>8}')

## Conclusions

### Experiment Summary

This notebook tested whether **longer prediction horizons** and **classification framing** can overcome the limitations observed in the next-day XGBoost experiment:

- **Horizons tested**: 5, 10, 20, 30 days ahead
- **Tasks**: Return regression + direction classification
- **Dimensionality reduction**: Removed near-zero variance and highly correlated features to address the curse of dimensionality
- **Hyperparameter optimization**: GridSearchCV with TimeSeriesSplit per horizon and task
- **Walk-forward validation**: Same expanding window, `TRAIN_PCT` initial training, 5-day step

### Key Findings

**Examine the results tables and trend plots above (Sections 7-9) to determine:**

- Whether longer horizons improve directional accuracy over the ~50% baseline
- Whether classification outperforms regression at predicting direction
- Which horizon yields the best signal-to-noise ratio
- Whether dimensionality reduction helped by reducing the feature-to-sample ratio
- How the optimal hyperparameters vary across horizons (Section 10)

### Comparison with Next-Day Experiment

The next-day XGBoost experiment showed:
- Directional accuracy ~50% (barely above random)
- More complexity = worse error metrics
- NLP features added noise

This multi-horizon experiment tests whether the signal becomes detectable at longer time scales, where the naive baseline weakens and trends become more pronounced.